<a href="https://colab.research.google.com/github/habibullah0101/TAPPY-BIRD/blob/main/multi_modal_food_and_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q ultralytics google-genai roboflow pillow
print("Dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 5.2 MB/s eta 0:00:00
Dependencies installed successfully!


In [2]:

#Download the Dataset (Nigeria-food-6)
from roboflow import Roboflow

# Initialize Roboflow (replace with your workspace/project info if needed)
rf = Roboflow(api_key="ktNE84r2f7lqUK65pUCF") # Or use your project download code
project = rf.workspace("nigeria-food").project("nigeria-food")
dataset = project.version(6).download("yolov8")

print("Dataset downloaded and ready in Colab!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Nigeria-food-6 in yolov8:: 100%|██████████| 22898/22898 [00:03<00:00, 6484.48it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Dataset downloaded and ready in Colab!


In [3]:
#Cell 3: Set Your Google Gemini API Key
import os
from google.colab import userdata

# Automatically loads your API key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
print("API Key configured!")

API Key configured!


In [4]:
#Cell 4: Run the Multimodal Processor Script
from PIL import Image
from google import genai

client = genai.Client()

def analyze_food_plate(image_path: str, user_text: str):
    image = Image.open(image_path)

    prompt = f"""
    You are an expert Nigerian nutritionist and AI food logging assistant.
    Analyze this food image and take into account the user's text note: "{user_text}".
    Output the result in clean JSON format with keys: food_items, estimated_portions, special_notes.
    """

    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[image, prompt],
    )
    return response.text

print("Multimodal function ready!")

Multimodal function ready!


In [5]:
from ultralytics import YOLO

# Load a pretrained lightweight YOLOv8 model
model = YOLO("yolov8n.pt")

print("Starting YOLO training on Google Colab T4 GPU...")

# Train using the dataset path automatically mapped by Roboflow in Colab
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    name="nigerian_food_colab"
)

print("Training finished successfully! Your model weights are saved.")


Starting YOLO training on Google Colab T4 GPU...
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Nigeria-food-6/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

In [6]:
#The Final Step: The Integration Pipeline

#final script. It takes a food image, runs your newly trained YOLO model to spot
# the dishes, and then passes both the image and YOLO's detections straight into Gemini to generate a clean JSON report:

from ultralytics import YOLO
from PIL import Image
from google import genai
import json

# 1. Load your newly trained custom YOLO weights
model_weights_path = "/content/runs/detect/nigerian_food_colab/weights/best.pt"
yolo_model = YOLO(model_weights_path)

# 2. Initialize Gemini client
gemini_client = genai.Client()

def analyze_nigerian_meal(image_path: str, user_text_note: str = ""):
    print(f"Running YOLO detection on {image_path}...")

    # Step A: Run YOLO detection on the image
    yolo_results = yolo_model(image_path)
    detected_classes = []

    for box in yolo_results[0].boxes:
        cls_id = int(box.cls[0])
        class_name = yolo_model.names[cls_id]
        detected_classes.append(class_name)

    # Remove duplicates
    detected_classes = list(set(detected_classes))
    print(f"YOLO detected: {detected_classes}")

    # Step B: Pass image, YOLO clues, and user text to Gemini for deep multimodal reasoning
    image = Image.open(image_path)

    prompt = f"""
    You are an expert Nigerian nutritionist and food logging AI.
    A user uploaded a food image with the text note: "{user_text_note}".
    Our computer vision model (YOLO) preliminarily detected these items in the image: {detected_classes}.

    Please analyze the image, verify or refine the food items based on visual evidence, estimate the portions, and output the result in clean, valid JSON format with the following keys:
    - "detected_food_items": list of strings
    - "estimated_portions": description or breakdown
    - "nutritional_insights": brief health notes tailored to Nigerian cuisine
    """

    print("Sending data to Gemini multimodal engine...")
    response = gemini_client.models.generate_content(
        model='gemini-2.5-flash',
        contents=[image, prompt],
    )

    return response.text

# Test the pipeline with one of your validation/test images!
# sample_image = "/content/Nigeria-food-6/test/images/your_test_image.jpg"
# print(analyze_nigerian_meal(sample_image, "I added extra palm oil and fried plantain."))
print("Integration pipeline function ready to deploy!")

Integration pipeline function ready to deploy!


In [8]:
#testing it with a real image

import os
import random

# Redefine the analyze_nigerian_meal function to use the updated model
# This assumes yolo_model and gemini_client are already initialized in the global scope
# from cell JuZ9AeaPd6zD.
def analyze_nigerian_meal(image_path: str, user_text_note: str = ""):
    print(f"Running YOLO detection on {image_path}...")

    # Step A: Run YOLO detection on the image
    yolo_results = yolo_model(image_path)
    detected_classes = []

    for box in yolo_results[0].boxes:
        cls_id = int(box.cls[0])
        class_name = yolo_model.names[cls_id]
        detected_classes.append(class_name)

    # Remove duplicates
    detected_classes = list(set(detected_classes))
    print(f"YOLO detected: {detected_classes}")

    # Step B: Pass image, YOLO clues, and user text to Gemini for deep multimodal reasoning
    image = Image.open(image_path)

    prompt = f"""
    You are an expert Nigerian nutritionist and food logging AI.
    A user uploaded a food image with the text note: "{user_text_note}".
    Our computer vision model (YOLO) preliminarily detected these items in the image: {detected_classes}.

    Please analyze the image, verify or refine the food items based on visual evidence, estimate the portions, and output the result in clean, valid JSON format with the following keys:
    - "detected_food_items": list of strings
    - "estimated_portions": description or breakdown
    - "nutritional_insights": brief health notes tailored to Nigerian cuisine
    """

    print("Sending data to Gemini multimodal engine...")
    response = gemini_client.models.generate_content(
        model='gemini-3.6-flash', # Updated model name
        contents=[image, prompt],
    )

    return response.text

# Grab a random image from your test folder
test_dir = f"{dataset.location}/test/images"
random_img_name = random.choice(os.listdir(test_dir))
sample_path = os.path.join(test_dir, random_img_name)

# Test the pipeline
output_json = analyze_nigerian_meal(
    image_path=sample_path,
    user_text_note="Added extra palm oil and fried plantain."
)

print("\n--- Final Multimodal Pipeline Output ---")
print(output_json)


Running YOLO detection on /content/Nigeria-food-6/test/images/Image_10_jpg.rf.ac3cbc41421e451ba78454c4364da856.jpg...

image 1/1 /content/Nigeria-food-6/test/images/Image_10_jpg.rf.ac3cbc41421e451ba78454c4364da856.jpg: 640x640 (no detections), 7.9ms
Speed: 1.3ms preprocess, 7.9ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
YOLO detected: []
Sending data to Gemini multimodal engine...

--- Final Multimodal Pipeline Output ---
```json
{
  "detected_food_items": [
    "Nigerian Moi Moi (Steamed Bean Pudding)",
    "Hard-boiled Egg",
    "Palm Oil (extra)",
    "Fried Plantain (Dodo)"
  ],
  "estimated_portions": "1 medium portion of Moi Moi (approx. 180g) garnished with half a hard-boiled egg (~25g), with approximately 1 tablespoon of extra palm oil (~14g) incorporated, plus a side serving of fried plantains (~75g) as noted.",
  "nutritional_insights": "Moi Moi is a highly nutritious, protein-rich dish made from peeled black-eyed peas, providing beneficial fiber, B-vi

In [9]:
# Option A: Download directly to your PC
from google.colab import files
files.download('/content/runs/detect/nigerian_food_colab/weights/best.pt')

# Option B: Save to your Google Drive (Recommended)
from google.colab import drive
drive.mount('/content/drive')
!cp /content/runs/detect/nigerian_food_colab/weights/best.pt /content/drive/MyDrive/nigerian_food_yolo_best.pt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mounted at /content/drive
